# SGOOTR CRC11 methods and results

This notebook fills in the CRC11 manuscript paragraph from the saved S10 state-specific fit and its fitted-tree segments. It does not rerun inference. The selected schedule has one NC->PT event and two PT->LN events.

The inference time coordinate has leaves at 0 and the root at the maximum time. For consistency with the figure's displayed tree coordinate, event times below are converted as `root_time - inference_time`. The notebook also checks the draft's stated total of 228 cells against the analyzed tree's labeled leaves.

In [3]:
import csv
import math
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display

FIT_RELATIVE_PATH = Path(
    "data/sgootr/all_trees/combined_inference/"
    "S10_CRC11_state_specific_3transition_shared_source_search/"
    "schedule_3857/inference.txt"
)
INPUT_RELATIVE_DIR = Path(
    "data/sgootr/all_trees/combined_inference/S10_CRC11_transition_chain_input"
)

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / FIT_RELATIVE_PATH).is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        f"Could not find {FIT_RELATIVE_PATH} from notebook directory {Path.cwd()}"
    )

fit_path = project_root / FIT_RELATIVE_PATH
input_dir = project_root / INPUT_RELATIVE_DIR
nodes_path = input_dir / "nodes_2.csv"
edges_path = input_dir / "edges_2.csv"
segments_path = fit_path.parent / "figures" / "CRC11_state_segments.csv"
for required_path in (nodes_path, edges_path, segments_path):
    if not required_path.is_file():
        raise FileNotFoundError(f"Required SGOOTR result file not found: {required_path}")

state_names = {0: "NC", 1: "PT", 2: "LN"}
parsed = {"transition_events": []}
reading_events = False
for raw_line in fit_path.read_text(encoding="utf-8").splitlines():
    line = raw_line.strip()
    if not line:
        continue
    if line.startswith("LL="):
        parsed["log_likelihood"] = float(line.split("=", 1)[1])
    elif line.startswith("birth:"):
        parsed["birth"] = [float(value) for value in line.split(":", 1)[1].split()]
    elif line.startswith("death:"):
        parsed["death"] = [float(value) for value in line.split(":", 1)[1].split()]
    elif line.startswith("sampling:"):
        parsed["sampling"] = [float(value) for value in line.split(":", 1)[1].split()]
    elif line == "transition_times_by_edge:":
        reading_events = True
    elif reading_events:
        fields = line.split()
        if len(fields) != 2 or fields[0] not in {"NC->PT", "PT->LN"}:
            raise ValueError(f"Unexpected transition-time row in {fit_path}: {line}")
        parsed["transition_events"].append(
            {"edge": fields[0], "inference_time": float(fields[1])}
        )

for parameter_name in ("birth", "death", "sampling"):
    values = parsed.get(parameter_name, [])
    if len(values) != len(state_names):
        raise ValueError(
            f"Expected three {parameter_name} values in {fit_path}; found {len(values)}"
        )
if len(parsed["transition_events"]) != 3:
    raise ValueError(
        f"Expected three named transition events in {fit_path}; "
        f"found {len(parsed['transition_events'])}"
    )
if Counter(event["edge"] for event in parsed["transition_events"]) != {
    "NC->PT": 1,
    "PT->LN": 2,
}:
    raise ValueError("The selected fit must contain one NC->PT and two PT->LN events.")

birth = dict(zip(state_names.values(), parsed["birth"], strict=True))
death = dict(zip(state_names.values(), parsed["death"], strict=True))
net_growth = {state: birth[state] - death[state] for state in state_names.values()}
relative_birth = {
    "PT/NC": birth["PT"] / birth["NC"],
    "LN/NC": birth["LN"] / birth["NC"],
}

with nodes_path.open(newline="", encoding="utf-8") as source:
    nodes = list(csv.DictReader(source))
with edges_path.open(newline="", encoding="utf-8") as source:
    tree_edges = list(csv.DictReader(source))
node_ids = {row["node_id"] for row in nodes}
parent_ids = {row["parent"] for row in tree_edges}
leaf_ids_from_edges = node_ids - parent_ids
leaf_rows = [row for row in nodes if row["is_leaf"].strip() in {"1", "True", "true"}]
leaf_ids_from_flags = {row["node_id"] for row in leaf_rows}
if leaf_ids_from_edges != leaf_ids_from_flags:
    raise ValueError("Leaf flags in nodes_2.csv do not match the edge-list topology.")
leaf_counts = Counter()
for row in leaf_rows:
    state_id = int(row["state_id"])
    if state_id not in state_names:
        raise ValueError(f"Unknown leaf state ID {state_id} in {nodes_path}")
    leaf_counts[state_names[state_id]] += 1
if sum(leaf_counts.values()) != len(leaf_ids_from_edges):
    raise ValueError("State-specific leaf counts do not sum to the analyzed tree total.")

with segments_path.open(newline="", encoding="utf-8") as source:
    raw_segments = list(csv.DictReader(source))
if not raw_segments:
    raise ValueError(f"No fitted tree segments found in {segments_path}")
root_time = max(
    max(float(row["segment_start_time"]), float(row["segment_end_time"]))
    for row in raw_segments
)
map_segments = {
    row["segment_id"]: row
    for row in raw_segments
    if row["is_most_likely_state"].strip().lower() == "true"
}
segments_by_edge = defaultdict(list)
for row in map_segments.values():
    segments_by_edge[row["original_edge"]].append(row)
for edge_segments in segments_by_edge.values():
    edge_segments.sort(key=lambda row: float(row["segment_start_time"]), reverse=True)

event_counts = Counter()
for edge_segments in segments_by_edge.values():
    for older, younger in zip(edge_segments, edge_segments[1:]):
        boundary = float(older["segment_end_time"])
        younger_start = float(younger["segment_start_time"])
        if not math.isclose(boundary, younger_start, rel_tol=0.0, abs_tol=1e-8):
            continue
        source_state = older["most_likely_state"]
        target_state = younger["most_likely_state"]
        if source_state == target_state:
            continue
        event_counts[(f"{source_state}->{target_state}", round(boundary, 8))] += 1

event_rows = []
for event in sorted(parsed["transition_events"], key=lambda item: item["inference_time"], reverse=True):
    inference_time = event["inference_time"]
    event_count = event_counts[(event["edge"], round(inference_time, 8))]
    event_rows.append(
        {
            "transition": event["edge"],
            "inference_time (leaves=0)": inference_time,
            "tree-panel time (root=0)": root_time - inference_time,
            "MAP transition-bearing branches": event_count,
        }
    )
ptln_rows = [row for row in event_rows if row["transition"] == "PT->LN"]
if [row["MAP transition-bearing branches"] for row in ptln_rows] != [1, 2]:
    raise ValueError(
        "The fitted tree segments do not show one then two PT->LN MAP transitions "
        "in root-to-leaf event order. Check that the selected fit and segment file match."
    )

expected_dataset_cells = 228
analyzed_tree_leaves = sum(leaf_counts.values())
count_discrepancy = analyzed_tree_leaves != expected_dataset_cells

print(f"Fit: {fit_path.relative_to(project_root)}")
print(f"Root-normalized log likelihood: {parsed['log_likelihood']:.6f}")
print(f"Analyzed tree leaves: {analyzed_tree_leaves}; draft dataset total: {expected_dataset_cells}")
if count_discrepancy:
    print(
        "WARNING: The selected inference tree has a different number of leaves "
        "than the draft's stated dataset total. The state counts below describe "
        "the analyzed tree and must not be presented as a partition of 228 until reconciled."
    )

parameter_table = pd.DataFrame(
    {
        "Birth rate": birth,
        "Death rate": death,
        "Net growth rate": net_growth,
    }
)
parameter_table.index.name = "State"
display(parameter_table.round(4))
display(pd.DataFrame([{"Ratio": key, "Relative birth rate": value} for key, value in relative_birth.items()]).round(4))
display(pd.DataFrame([{"State": state, "Leaves in fitted tree": leaf_counts[state]} for state in state_names.values()]))
display(pd.DataFrame(event_rows).round(6))

Fit: data/sgootr/all_trees/combined_inference/S10_CRC11_state_specific_3transition_shared_source_search/schedule_3857/inference.txt
Root-normalized log likelihood: 327.294240
Analyzed tree leaves: 227; draft dataset total: 228


,Birth rate,Death rate,Net growth rate
State,,,
NC,27.5352,0.6835,26.8517
PT,33.6848,0.7408,32.9440
LN,30.2272,0.8072,29.4200


,Ratio,Relative birth rate
0,PT/NC,1.2233
1,LN/NC,1.0978


,State,Leaves in fitted tree
0,NC,25
1,PT,134
2,LN,68


,transition,inference_time (leaves=0),tree-panel time (root=0),MAP transition-bearing branches
0,NC->PT,0.356,0.087103,1
1,PT->LN,0.282,0.161103,1
2,PT->LN,0.131,0.312103,2


In [4]:
def number(value):
    return f"{value:.2f}"

first_ptln = ptln_rows[0]
second_ptln = ptln_rows[1]
count_note = (
    f"The source description gives 228 cells, whereas the selected fitted tree "
    f"contains {analyzed_tree_leaves} labeled leaves ({leaf_counts['NC']} NC, "
    f"{leaf_counts['PT']} PT, and {leaf_counts['LN']} LN); reconcile this "
    f"difference before treating the state counts as a partition of the full dataset."
    if count_discrepancy
    else f"The analyzed tree contains {analyzed_tree_leaves} labeled leaves: "
         f"{leaf_counts['NC']} NC, {leaf_counts['PT']} PT, and {leaf_counts['LN']} LN."
)

result_text = rf'''We applied \ourmethod to single-cell DNA methylation data from colorectal cancer patient CRC11 profiled by Bian et al.~\cite{{bian2018single}}. {count_note}
We reconstructed an ultrametric lineage tree using UPGMA on pairwise methylation-based distances estimated by Sgootr~\cite{{liu2023single}}. Following the sequential-progression model considered by Liu et al.~\cite{{liu2023single}}, we assumed a linear transition graph from non-cancer to primary-tumor to lymph-node states. The first transition represents malignant progression, whereas the second represents metastatic migration.
Under this model, our identifiability results allow joint inference of effective transition times, net growth rates, and relative birth rates without knowing the absolute sampling probabilities.

\ourmethod reveals two metastatic seeding events. In root-to-leaf order, the first involved one MAP PT-to-LN transition-bearing branch at tree-panel time {number(first_ptln['tree-panel time (root=0)'])}, and the second involved two such branches at tree-panel time {number(second_ptln['tree-panel time (root=0)'])}. These plotted tree times are transformed from the fit's leaf-zero inference coordinates by subtracting each fitted time from the root time.
The inferred net growth rates for non-cancer, primary-tumor, and lymph-node cells were {number(net_growth['NC'])}, {number(net_growth['PT'])}, and {number(net_growth['LN'])}, respectively (per unit time). Relative birth rates showed the same ordering: primary-tumor and lymph-node cells divided {number(relative_birth['PT/NC'])}-fold and {number(relative_birth['LN/NC'])}-fold faster than non-cancer cells, respectively.
These point estimates suggest that metastatic dissemination does not necessarily confer a growth advantage: primary-tumor cells exhibited higher division and net growth rates than cells in the lymph-node metastasis.'''

print(result_text)
print("\nInterpretation note: event-branch counts are based on the fitted tree's most-likely state segments.")

We applied \ourmethod to single-cell DNA methylation data from colorectal cancer patient CRC11 profiled by Bian et al.~\cite{bian2018single}. The source description gives 228 cells, whereas the selected fitted tree contains 227 labeled leaves (25 NC, 134 PT, and 68 LN); reconcile this difference before treating the state counts as a partition of the full dataset.
We reconstructed an ultrametric lineage tree using UPGMA on pairwise methylation-based distances estimated by Sgootr~\cite{liu2023single}. Following the sequential-progression model considered by Liu et al.~\cite{liu2023single}, we assumed a linear transition graph from non-cancer to primary-tumor to lymph-node states. The first transition represents malignant progression, whereas the second represents metastatic migration.
Under this model, our identifiability results allow joint inference of effective transition times, net growth rates, and relative birth rates without knowing the absolute sampling probabilities.

\ourmethod

## Transition-count comparison using AIC

This post hoc comparison uses the saved best log-likelihood for each candidate number of PT→LN transition times; it does not rerun inference. The fit has 11 common free parameters (three state-specific birth rates, three death rates, three sampling probabilities, and two source-state transition probabilities). Each model also estimates one NC→PT time and K PT→LN times, so the nominal parameter count is p_K = 11 + 1 + K = 12 + K.

$$
\mathrm{AIC}_K = -2\ell_K + 2p_K, \qquad \Delta \mathrm{AIC}_K = \mathrm{AIC}_K - \min_j \mathrm{AIC}_j.
$$

| PT→LN times | Parameters | Log-likelihood | AIC | ΔAIC (from best) |
|---:|---:|---:|---:|---:|
| 1 | 13 | 290.555649 | -555.111299 | 71.477181 |
| 2 | 14 | 327.294240 | -626.588480 | 0.000000 |
| 3 | 15 | 327.108987 | -624.217975 | 2.370505 |
| 4 | 16 | 328.125035 | -624.250071 | 2.338409 |
| 5 | 17 | 328.345254 | -622.690509 | 3.897971 |


The table reports both AIC and ΔAIC relative to the minimum AIC across the candidate counts (not relative to K=2). This is an exploratory ranking, not a significance test: ordinary AIC assumptions may be imperfect for change-point times, and the saved values are the best fits found by separate schedule searches rather than guaranteed global maxima.

In [ ]:
from pathlib import Path

import math
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

COUNT_RELATIVE_PATH = Path(
    "data/sgootr/all_trees/combined_inference/"
    "S10_CRC11_ptln_transition_count_analysis/"
    "likelihood_by_ptln_transition_count.csv"
)
count_project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / COUNT_RELATIVE_PATH).is_file()
    ),
    None,
)
if count_project_root is None:
    raise FileNotFoundError(
        f"Could not find {COUNT_RELATIVE_PATH} from {Path.cwd()}"
    )

count_results = pd.read_csv(count_project_root / COUNT_RELATIVE_PATH)
required_columns = {"num_ptln_transition_times", "num_ncpt_transition_times", "log_likelihood"}
if not required_columns.issubset(count_results.columns):
    raise ValueError(f"Missing columns in transition-count results: {required_columns - set(count_results.columns)}")
if count_results["num_ncpt_transition_times"].ne(1).any():
    raise ValueError("Expected exactly one NC→PT transition time in every candidate.")
if count_results["num_ptln_transition_times"].duplicated().any():
    raise ValueError("Expected one saved likelihood per PT→LN transition count.")
if not count_results["log_likelihood"].map(math.isfinite).all():
    raise ValueError("All saved log-likelihoods must be finite.")

count_results = count_results.sort_values("num_ptln_transition_times").reset_index(drop=True)
counts = count_results["num_ptln_transition_times"].astype(int)
if counts.tolist() != list(range(1, len(counts) + 1)) or 2 not in counts.tolist():
    raise ValueError("Expected contiguous PT→LN counts starting at one, including count two.")

count_results["n_parameters"] = 11 + 1 + counts
count_results["aic"] = (
    -2 * count_results["log_likelihood"]
    + 2 * count_results["n_parameters"]
)
count_results["delta_aic"] = count_results["aic"] - count_results["aic"].min()
preferred_count = int(
    count_results.loc[count_results["delta_aic"].idxmin(), "num_ptln_transition_times"]
)
display(
    count_results[
        ["num_ptln_transition_times", "n_parameters", "log_likelihood", "aic", "delta_aic"]
    ].rename(
        columns={
            "num_ptln_transition_times": "PT→LN times",
            "n_parameters": "Parameters",
            "log_likelihood": "Best log-likelihood",
            "aic": "AIC",
            "delta_aic": "ΔAIC (from best)",
        }
    ).round({"Best log-likelihood": 3, "AIC": 3, "ΔAIC (from best)": 3})
)
print(f"Minimum-AIC choice: K={preferred_count} PT→LN transition times.")

colors = ["#e5a000" if count == preferred_count else "#4c72b0" for count in counts]
figure, axis = plt.subplots(figsize=(7, 4))
axis.bar(counts, count_results["delta_aic"], color=colors)
axis.set_xticks(counts)
axis.set_xlabel("Number of PT→LN transition times")
axis.set_ylabel("ΔAIC (from minimum; lower is better)")
axis.set_title("CRC11 transition-count comparison")
axis.axhline(0, color="#555555", linewidth=0.8)
figure.tight_layout()
plt.show()
